In [26]:
import pandas as pd
import numpy as np


def manual_excel_baseline_forecast(
    df: pd.DataFrame,
    target_month: str,
    eic_col: str = "eic_code",
    target_col: str = "sum_of_kWh",
    datetime_col: str = "datetime",
    day_col: str = "day",
    hour_col: str = "hour",
) -> pd.DataFrame:

    data = df.copy()
    data[datetime_col] = pd.to_datetime(data[datetime_col])
    tz = data[datetime_col].dt.tz  # automatically detect (Europe/Kyiv)

    target_start = pd.Period(target_month, freq="M").start_time.tz_localize(tz)
    target_end = pd.Period(target_month, freq="M").end_time.tz_localize(tz)

    prev_month = target_start - pd.DateOffset(months=1)
    same_month_last_year = target_start - pd.DateOffset(years=1)
    prev_month_last_year = prev_month - pd.DateOffset(years=1)

    data["month"] = data[datetime_col].dt.to_period("M")
    data[day_col] = data[datetime_col].dt.day
    data[hour_col] = data[datetime_col].dt.hour

    prev_period = prev_month.to_period("M")
    same_ly_period = same_month_last_year.to_period("M")
    prev_ly_period = prev_month_last_year.to_period("M")

    # --- Previous month template ---
    prev_template = (
        data[data["month"] == prev_period]
        .groupby([eic_col, day_col, hour_col], as_index=False)[target_col]
        .mean()
        .rename(columns={target_col: "previous_month_kWh"})
    )

    # --- Growth factor ---
    same_ly = data[data["month"] == same_ly_period]
    prev_ly = data[data["month"] == prev_ly_period]

    same_ly_mean = same_ly.groupby(eic_col)[target_col].mean()
    prev_ly_mean = prev_ly.groupby(eic_col)[target_col].mean()

    growth = (
        pd.concat([same_ly_mean, prev_ly_mean], axis=1)
        .rename(columns={
            target_col: "same_month_last_year",
            target_col: "previous_month_last_year"
        })
    )

    growth.columns = ["same_month_last_year", "previous_month_last_year"]

    growth["yoy_growth_factor"] = np.where(
        growth["previous_month_last_year"] > 0,
        growth["same_month_last_year"] / growth["previous_month_last_year"],
        1.0,
    )

    growth = growth[["yoy_growth_factor"]].reset_index()

    # --- Forecast calendar ---
    forecast_calendar = pd.DataFrame({
        datetime_col: pd.date_range(
            start=target_start,
            end=target_end.floor("h"),
            freq="h"
        )
    })
    forecast_calendar[day_col] = forecast_calendar[datetime_col].dt.day
    forecast_calendar[hour_col] = forecast_calendar[datetime_col].dt.hour

    stations = data[eic_col].dropna().unique()

    forecast_grid = (
        pd.MultiIndex.from_product(
            [stations, forecast_calendar[datetime_col]],
            names=[eic_col, datetime_col]
        )
        .to_frame(index=False)
    )

    forecast_grid[day_col] = forecast_grid[datetime_col].dt.day
    forecast_grid[hour_col] = forecast_grid[datetime_col].dt.hour

    # --- Merge template + growth ---
    out = forecast_grid.merge(
        prev_template,
        on=[eic_col, day_col, hour_col],
        how="left"
    )

    out = out.merge(growth, on=eic_col, how="left")
    out["yoy_growth_factor"] = out["yoy_growth_factor"].fillna(1.0)

    out["baseline_forecast_kWh"] = (
        out["previous_month_kWh"] * out["yoy_growth_factor"]
    )

    # --- Attach REAL target values ---
    actuals = data[
        (data[datetime_col] >= target_start) &
        (data[datetime_col] <= target_end)
    ][[eic_col, datetime_col, target_col, "dam_price", "sell_bm_price", "buy_bm_price"]]

    out = out.merge(
        actuals,
        on=[eic_col, datetime_col],
        how="left"
    )

    return out.sort_values([eic_col, datetime_col]).dropna().reset_index(drop=True)

In [27]:
df = pd.read_parquet("data/silver/feature_engineered_built_idx_split.parquet")
df

,datetime,eic_code,sum_of_kWh,max_power,max_solar,max_ev,latitude,longitude,dso_desc,station_type,...,Month,Day,Hour,day_of_week,season,time_idx,sell_bm_price,buy_bm_price,dam_price,data_subset
0,2024-01-01 01:00:00+02:00,62Z0008583037334,24.043645,110.0,25.52,0.0,48.442430,22.192190,Ужгород,ОККО-комплекс,...,1,1,1,0,1,0,0.01,59.85,57.0,train
1,2024-01-01 01:00:00+02:00,62Z0011230718431,16.720749,52.0,0.00,0.0,51.542106,31.262997,Чернігів,ОККО-міська,...,1,1,1,0,1,0,0.01,59.85,57.0,train
2,2024-01-01 01:00:00+02:00,62Z0096677872985,14.889751,35.0,15.00,0.0,48.569660,22.346380,Ужгород,ОККО-трасова,...,1,1,1,0,1,0,0.01,59.85,57.0,train
3,2024-01-01 01:00:00+02:00,62Z013852333354Y,44.286251,514.0,50.00,270.0,48.566059,30.230684,Черкаси,ОККО-комплекс,...,1,1,1,0,1,0,0.01,59.85,57.0,train
4,2024-01-01 01:00:00+02:00,62Z0211989286227,16.102043,210.0,0.00,0.0,48.279279,26.055273,Чернівці,ОККО-трасова,...,1,1,1,0,1,0,0.01,59.85,57.0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5175456,2025-09-01 03:00:00+03:00,62Z9807667180062,15.496269,85.0,0.00,0.0,48.876403,25.008546,Франківськ,ОККО-міні,...,9,1,3,0,4,14615,0.01,829.50,790.0,test
5175457,2025-09-01 03:00:00+03:00,62Z9964019189968,17.007319,80.0,20.00,0.0,49.861099,24.020287,Львів,ОККО-міська,...,9,1,3,0,4,14615,0.01,829.50,790.0,test
5175458,2025-09-01 03:00:00+03:00,62Z9967272139607,1.685111,10.0,0.00,0.0,49.514413,34.428297,Полтава,ОККО-міні,...,9,1,3,0,4,14615,0.01,829.50,790.0,test
5175459,2025-09-01 03:00:00+03:00,62Z9972124228553,82.799274,320.0,39.15,270.0,50.419477,29.842368,КиївОбл,ОККО-комплекс,...,9,1,3,0,4,14615,0.01,829.50,790.0,test


In [28]:
val_eval = manual_excel_baseline_forecast(
    df,
    target_month="2025-07",
    eic_col = "eic_code",
    target_col = "sum_of_kWh",
    datetime_col = "datetime",
    day_col = "Day",
    hour_col = "Hour",
)

test_eval = manual_excel_baseline_forecast(
    df,
    target_month="2025-08",
    eic_col = "eic_code",
    target_col = "sum_of_kWh",
    datetime_col = "datetime",
    day_col = "Day",
    hour_col = "Hour",
)

C:\Users\Lev\AppData\Local\Temp\ipykernel_15372\2737079252.py:26: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data["month"] = data[datetime_col].dt.to_period("M")
C:\Users\Lev\AppData\Local\Temp\ipykernel_15372\2737079252.py:30: UserWarning: Converting to Period representation will drop timezone information.
  prev_period = prev_month.to_period("M")
C:\Users\Lev\AppData\Local\Temp\ipykernel_15372\2737079252.py:31: UserWarning: Converting to Period representation will drop timezone information.
  same_ly_period = same_month_last_year.to_period("M")
C:\Users\Lev\AppData\Local\Temp\ipykernel_15372\2737079252.py:32: UserWarning: Converting to Period representation will drop timezone information.
  prev_ly_period = prev_month_last_year.to_period("M")
C:\Users\Lev\AppData\Local\Temp\ipykernel_15372\2737079252.py:26: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data["month"] = data[datetime_col]

In [29]:
from training.loss_funcs import *

In [30]:
val_eval

,eic_code,datetime,Day,Hour,previous_month_kWh,yoy_growth_factor,baseline_forecast_kWh,sum_of_kWh,dam_price,sell_bm_price,buy_bm_price
0,62Z0008583037334,2025-07-01 00:00:00+03:00,1,0,19.000000,0.994559,18.896624,21.000000,5568.52,0.01,5846.95
1,62Z0008583037334,2025-07-01 01:00:00+03:00,1,1,18.000000,0.994559,17.902065,19.000000,5568.42,0.01,5846.84
2,62Z0008583037334,2025-07-01 02:00:00+03:00,1,2,15.000000,0.994559,14.918387,16.000000,5190.00,0.01,5449.50
3,62Z0008583037334,2025-07-01 03:00:00+03:00,1,3,15.000000,0.994559,14.918387,16.000000,4888.00,0.01,5132.40
4,62Z0008583037334,2025-07-01 04:00:00+03:00,1,4,15.000000,0.994559,14.918387,15.000000,4299.00,0.01,4513.95
...,...,...,...,...,...,...,...,...,...,...,...
284395,62Z9997819406173,2025-07-30 19:00:00+03:00,30,19,22.187634,0.860210,19.086016,22.612939,8700.00,8265.00,10489.50
284396,62Z9997819406173,2025-07-30 20:00:00+03:00,30,20,21.898813,0.860210,18.837569,22.799174,9000.00,8550.00,10494.61
284397,62Z9997819406173,2025-07-30 21:00:00+03:00,30,21,21.909128,0.860210,18.846443,22.759967,9000.00,4.62,9450.00
284398,62Z9997819406173,2025-07-30 22:00:00+03:00,30,22,22.032908,0.860210,18.952920,23.318674,8888.19,4.52,9332.60


In [31]:
Y_COL = "sum_of_kWh"

def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

val_smape_v     = smape(val_eval[Y_COL], val_eval['baseline_forecast_kWh'])
val_rmse_v      = rmse(val_eval[Y_COL], val_eval['baseline_forecast_kWh'])
val_mape_v      = mape(val_eval[Y_COL], val_eval['baseline_forecast_kWh'])
val_money_v     = money(val_eval[Y_COL], val_eval['baseline_forecast_kWh'], *_prices(val_eval))
val_money_pct_v = money_pct(val_eval[Y_COL], val_eval['baseline_forecast_kWh'], *_prices(val_eval))

test_smape_v     = smape(test_eval[Y_COL], test_eval['baseline_forecast_kWh'])
test_rmse_v      = rmse(test_eval[Y_COL], test_eval['baseline_forecast_kWh'])
test_mape_v      = mape(test_eval[Y_COL], test_eval['baseline_forecast_kWh'])
test_money_v     = money(test_eval[Y_COL], test_eval['baseline_forecast_kWh'], *_prices(test_eval))
test_money_pct_v = money_pct(test_eval[Y_COL], test_eval['baseline_forecast_kWh'], *_prices(test_eval))

print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE     : {val_smape_v:.4f}")
print(f"RMSE      : {val_rmse_v:.4f}")
print(f"MAPE      : {val_mape_v:.2f} %")
print(f"MONEY     : {val_money_v:.4f}")
print(f"MONEY_PCT : {val_money_pct_v:.4f}%")

print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE     : {test_smape_v:.4f}")
print(f"RMSE      : {test_rmse_v:.4f}")
print(f"MAPE      : {test_mape_v:.2f} %")
print(f"MONEY     : {test_money_v:.4f}")
print(f"MONEY_PCT : {test_money_pct_v:.4f}%")

── Validation ──────────────────────────────────────────────
Aligned samples : 284,400
SMAPE     : 0.2539
RMSE      : 10.0837
MAPE      : 27.72 %
MONEY     : 2503332.3831
MONEY_PCT : 8.8381%
── Test ────────────────────────────────────────────────────
Aligned samples : 293,880
SMAPE     : 0.2911
RMSE      : 28.3564
MAPE      : 55.53 %
MONEY     : 7840106.3907
MONEY_PCT : 26.6080%


In [32]:
june_eval = manual_excel_baseline_forecast(
    df,
    target_month="2025-06",
    eic_col = "eic_code",
    target_col = "sum_of_kWh",
    datetime_col = "datetime",
    day_col = "Day",
    hour_col = "Hour",
)

june_mape_v      = mape(june_eval[Y_COL], june_eval['baseline_forecast_kWh'])
june_money_v     = money(june_eval[Y_COL], june_eval['baseline_forecast_kWh'], *_prices(june_eval))
june_money_pct_v = money_pct(june_eval[Y_COL], june_eval['baseline_forecast_kWh'], *_prices(june_eval))

print("── June ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"MAPE      : {june_mape_v:.2f} %")
print(f"MONEY     : {june_money_v:.4f}")
print(f"MONEY_PCT : {june_money_pct_v:.4f}%")

C:\Users\Lev\AppData\Local\Temp\ipykernel_15372\2737079252.py:26: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data["month"] = data[datetime_col].dt.to_period("M")
C:\Users\Lev\AppData\Local\Temp\ipykernel_15372\2737079252.py:30: UserWarning: Converting to Period representation will drop timezone information.
  prev_period = prev_month.to_period("M")
C:\Users\Lev\AppData\Local\Temp\ipykernel_15372\2737079252.py:31: UserWarning: Converting to Period representation will drop timezone information.
  same_ly_period = same_month_last_year.to_period("M")
C:\Users\Lev\AppData\Local\Temp\ipykernel_15372\2737079252.py:32: UserWarning: Converting to Period representation will drop timezone information.
  prev_ly_period = prev_month_last_year.to_period("M")


── June ──────────────────────────────────────────────
Aligned samples : 284,400
MAPE      : 52.41 %
MONEY     : 3511502.1332
MONEY_PCT : 15.1352%
